# NSE Liquidity Sweep Screener — Swing-Buy Setups
### Daily sweep of swing lows (old or new) with close reclaim + proper wick · NSE Top 1000 by turnover

This notebook scans the **Top 1000 most liquid NSE stocks** (ranked by average daily turnover, in ₹ Cr)
for a specific institutional footprint on the **latest daily candle**:

| # | Rule (hard filter) | Meaning |
|---|--------------------|---------|
| 1 | A **swing low** (5-bar fractal) exists within the last `level_lookback` bars — old or new | that level = resting liquidity (stop-losses of longs below it) |
| 2 | Today's candle **LOW pierces below** the level (min 0.05%) | the sweep actually happened — liquidity got taken |
| 3 | Candle **CLOSE is above** the level (min +0.1% buffer) | sweep is *complete* — price rejected back above |
| 4 | Lower wick ≥ **30% of full range** and ≥ 0.15% of price | "proper wick" — visible rejection, not noise |
| 5 | Wick runs **no deeper than 2.5%** below the level | deep wick = breakdown, not a sweep |
| 6 | Close > open (bullish candle) | cleaner long entry |
| 7 | Previous close was **above** the level | true sweep *from above*, not a continuation through a broken level |

Every setup is then **scored 0–100** on wick quality, close strength, sweep depth, volume confirmation,
RSI context and EMA trend context, and the table is sorted by score.

**Pipeline:** NSE full stock list → rank by turnover → Top 1000 → robust chunked download (retries +
backoff) → data-health checks (staleness / short history / OHLC repair) → sweep engine → scored results +
charts for the top picks.

**How to run (Google Colab):** upload this `.ipynb` → open in Colab → *Runtime ▸ Run all*.
Typical runtime **5–10 minutes** (most of it is data download). For confirmed sweeps, run **after
15:35 IST** (after the NSE close) — intraday runs will include the live, incomplete candle.

> ⚠️ **Disclaimer:** educational research tool. Not investment advice. Liquidity sweeps can fail —
> always risk a fixed % of capital, stop below the sweep low.

In [ ]:
# ============================================================
# 0. SETUP — dependencies (Colab-safe, idempotent)
# ============================================================
import sys, subprocess, importlib

def _ensure(pkg, mod=None):
    try:
        importlib.import_module(mod or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

_ensure("yfinance")
_ensure("requests")

import warnings
warnings.filterwarnings("ignore")

import time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 240)
print(f"yfinance {yf.__version__} | pandas {pd.__version__} | numpy {np.__version__}  —  setup OK")

## 1. Configuration
All knobs in one place. Tune here, then re-run the cells below.

In [ ]:
# ============================================================
# 1. CONFIGURATION — tune the screener here
# ============================================================
from dataclasses import dataclass

@dataclass
class SweepConfig:
    # ---- Universe ----
    universe_size       : int   = 1000     # NSE top-N by avg daily turnover
    min_avg_turnover_cr : float = 0.25     # min ₹ Cr avg daily turnover to stay in universe
    rank_bars_needed    : int   = 8        # min bars to be eligible for turnover ranking
    # ---- Data ----
    history_period      : str   = "1y"     # ~250 daily bars of history
    min_bars            : int   = 60       # drop tickers with thinner history
    require_latest_session : bool = True   # screen ONLY candles of the market's latest session
    backfill_latest_close  : bool = True   # rebuild a missing latest-session candle from 15-min data
    max_staleness_days  : int   = 4        # (used only if require_latest_session=False)
    download_chunk      : int   = 80       # symbols per Yahoo request
    max_retries         : int   = 3        # retries per chunk
    backoff_secs        : float = 4.0      # base backoff (doubles per retry)
    # ---- Swing low (liquidity level) detection ----
    fractal_k           : int   = 2        # 5-bar fractal: low = min of [i-2 .. i+2]
    level_lookback      : int   = 120      # level may be up to N bars old (old OR new)
    level_dedup_pct     : float = 0.0015   # levels within 0.15% = same liquidity pool
    # ---- Sweep conditions (hard filters on latest candle) ----
    min_pierce_pct      : float = 0.0005   # low must pierce >= 0.05% below level
    close_above_buffer  : float = 0.001    # close must be >= 0.1% above level
    max_sweep_depth     : float = 0.025    # wick deeper than 2.5% below level = breakdown, reject
    min_wick_ratio      : float = 0.30     # lower wick >= 30% of candle range ("proper wick")
    min_wick_pct_price  : float = 0.0015   # and >= 0.15% of price (ignore micro-noise)
    require_bullish_close : bool = True    # close > open
    require_prior_above : bool = True      # previous close above the level (true sweep)
    min_price           : float = 10.0     # ignore sub-₹10 names
    # ---- Scoring ----
    volume_lookback     : int   = 20
    rsi_period          : int   = 14

CFG = SweepConfig()

print("Active rules on the LATEST daily candle")
print("-" * 62)
print(f"  Swing low           : 5-bar fractal, up to {CFG.level_lookback} bars old (old or new)")
print(f"  Sweep pierce        : low >= {CFG.min_pierce_pct:.2%} below level, no deeper than {CFG.max_sweep_depth:.1%}")
print(f"  Reclaim             : close >= {CFG.close_above_buffer:.2%} ABOVE level")
print(f"  Proper wick         : lower wick >= {CFG.min_wick_ratio:.0%} of range AND >= {CFG.min_wick_pct_price:.2%} of price")
print(f"  Bullish close       : {CFG.require_bullish_close}   |   Prior close above level: {CFG.require_prior_above}")
print(f"  Universe            : top {CFG.universe_size} by turnover (>= ₹{CFG.min_avg_turnover_cr} Cr/day), price >= ₹{CFG.min_price:.0f}")

## 2. Universe — full NSE stock list
Primary source: **NSE official archive** (`nsearchives.nseindia.com` EQUITY_L.csv — no cookies needed).
If NSE blocks the request (happens on some datacenter IPs), an embedded fallback list of the
~300 most liquid NSE names is used automatically.

In [ ]:
# ============================================================
# 2. UNIVERSE — full NSE stock list (official archive + fallback)
# ============================================================
import io, requests

NSE_LIST_URLS = [
    "https://nsearchives.nseindia.com/content/equities/EQUITY_L.csv",
    "https://archives.nseindia.com/content/equities/EQUITY_L.csv",   # legacy mirror
]
HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                   "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"),
    "Accept": "text/csv,application/csv,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.nseindia.com/",
}

# Fallback: ~top 300 liquid NSE names (used only if the official list is unreachable)
FALLBACK_SYMBOLS = [
    "BSE",
    "HDFCBANK",
    "DHOOTTRANS",
    "NETWEB",
    "HINDCOPPER",
    "ICICIBANK",
    "RELIANCE",
    "GROWW",
    "BHARTIARTL",
    "WELCORP",
    "ETERNAL",
    "INFY",
    "SHIPROCKET",
    "MCX",
    "ASTERDM",
    "CUPID",
    "HSCL",
    "MILKYMIST",
    "PAYTM",
    "SBIN",
    "DIXON",
    "LENSKART",
    "BALRAMCHIN",
    "MOLBIO",
    "SAIL",
    "KOTAKBANK",
    "IDEA",
    "TCS",
    "ATHERENERG",
    "AXISBANK",
    "BLEL",
    "HINDZINC",
    "DATAPATTNS",
    "TATASTEEL",
    "LT",
    "M&M",
    "KALYANKJIL",
    "LICI",
    "KAYNES",
    "COFORGE",
    "LAURUSLABS",
    "VIYASH",
    "IFCI",
    "BAJFINANCE",
    "ADANIPOWER",
    "HFCL",
    "ADANIENSOL",
    "GRASIM",
    "COALINDIA",
    "GVT&D",
    "VEDL",
    "HAL",
    "BEL",
    "MUTHOOTFIN",
    "DIVISLAB",
    "TVSMOTOR",
    "ITC",
    "ADANIENT",
    "BHEL",
    "APOLLOHOSP",
    "MANORAMA",
    "PCJEWELLER",
    "MVELECTRO",
    "VBL",
    "FMGOETZE",
    "FACT",
    "INDOMIM",
    "TMCV",
    "MEESHO",
    "SWIGGY",
    "URBANCO",
    "MARUTI",
    "KWIL",
    "SHRIRAMFIN",
    "HINDALCO",
    "PWL",
    "PARAS",
    "LGEINDIA",
    "VAML",
    "ONGC",
    "TECHM",
    "PFC",
    "ZEEL",
    "ADANIPORTS",
    "VMM",
    "BALUFORGE",
    "CYIENT",
    "TEJASNET",
    "POWERINDIA",
    "TMPV",
    "ULTRACEMCO",
    "PINELABS",
    "CHENNPETRO",
    "SUNPHARMA",
    "ICICIAMC",
    "POLYCAB",
    "APARINDS",
    "HDFCAMC",
    "AMBER",
    "CUMMINSIND",
    "BAJAJHIND",
    "SOLARINDS",
    "CGCL",
    "INDIGO",
    "TITAN",
    "BDL",
    "JSWSTEEL",
    "RATNAVEER",
    "PERSISTENT",
    "NTPC",
    "BBTC",
    "MOREPENLAB",
    "EICHERMOT",
    "BAJAJ-AUTO",
    "OFSS",
    "ASHOKLEY",
    "CDSL",
    "NATIONALUM",
    "TFCILTD",
    "HEROMOTOCO",
    "LEAPIND",
    "HCLTECH",
    "IIFL",
    "POLICYBZR",
    "LTFOODS",
    "MRPL",
    "NESTLEIND",
    "REDINGTON",
    "ZAGGLE",
    "WELSPUNLIV",
    "HINDUNILVR",
    "POWERGRID",
    "MANAPPURAM",
    "TATAPOWER",
    "VOGL",
    "MAZDOCK",
    "JIOFIN",
    "IPCALAB",
    "CGPOWER",
    "WEL",
    "DMART",
    "AUBANK",
    "TIINDIA",
    "VIKRAMSOLR",
    "DLF",
    "AZAD",
    "BOSCHLTD",
    "LICHSGFIN",
    "SUZLON",
    "NMDC",
    "TDPOWERSYS",
    "ICICIGI",
    "FEDERALBNK",
    "BHARATFORG",
    "MOTILALOFS",
    "NAUKRI",
    "QUADFUTURE",
    "MAXHEALTH",
    "APOLLO",
    "IDBI",
    "RECLTD",
    "BANKBARODA",
    "WABAG",
    "KEI",
    "MOTHERSON",
    "HYUNDAI",
    "SIEMENS",
    "DRREDDY",
    "TECHNOCRAF",
    "LTF",
    "ADANIGREEN",
    "SHADOWFAX",
    "HONASA",
    "UNIONBANK",
    "COCHINSHIP",
    "TRENT",
    "SONACOMS",
    "LODHA",
    "INDUSTOWER",
    "CARTRADE",
    "ANGELONE",
    "LUPIN",
    "JYOTICNC",
    "LTM",
    "BRITANNIA",
    "ZYDUSLIFE",
    "BPCL",
    "JINDRILL",
    "GENUSPOWER",
    "NAM-INDIA",
    "VOLTAS",
    "PTCIL",
    "PAGEIND",
    "CHOLAFIN",
    "PARADEEP",
    "SBILIFE",
    "ENRIN",
    "HINDPETRO",
    "ICIL",
    "SBICARD",
    "SYRMA",
    "CANBK",
    "JUBLFOOD",
    "KFINTECH",
    "OLAELEC",
    "HDFCLIFE",
    "TORNTPHARM",
    "MANIPALHOS",
    "NYKAA",
    "SAMMAANCAP",
    "ASIANPAINT",
    "WIPRO",
    "HCC",
    "SAGILITY",
    "GESHIP",
    "PREMIERENE",
    "JINDALSAW",
    "HEG",
    "AVALON",
    "RUBICON",
    "ZENTEC",
    "GLENMARK",
    "HAPPSTMNDS",
    "GRSE",
    "CROMPTON",
    "MAHABANK",
    "RADICO",
    "VISL",
    "ABCAPITAL",
    "AUROPHARMA",
    "BAJAJFINSV",
    "AEGISLOG",
    "BLS",
    "MOTISONS",
    "GODREJCP",
    "PNB",
    "APLAPOLLO",
    "INOXINDIA",
    "ABB",
    "OBEROIRLTY",
    "KERNEX",
    "WAAREEENER",
    "RENUKA",
    "SBIFUNDS",
    "BELRISE",
    "RRKABEL",
    "TATACONSUM",
    "AMBUJACEM",
    "STYLEBAAZA",
    "WOCKPHARMA",
    "PIDILITIND",
    "GLAND",
    "FINCABLES",
    "YESBANK",
    "UPL",
    "KTKBANK",
    "RAMBHAJO",
    "GODREJPROP",
    "NEULANDLAB",
    "SUDARSCHEM",
    "EXIDEIND",
    "ANANTRAJ",
    "GAIL",
    "GMRAIRPORT",
    "MARKSANS",
    "SHILPAMED",
    "JBMA",
    "JINDALSTEL",
    "EIDPARRY",
    "ACMESOLAR",
    "CPPLUS",
    "IOC",
    "ORIENTHOT",
    "BANDHANBNK",
    "CAMS",
    "FORTIS",
    "INDUSINDBK",
    "DCBBANK",
    "CIPLA",
    "STLTECH",
    "KPITTECH",
    "NAVINFLUOR",
    "IDFCFIRSTB",
    "FORCEMOT",
    "BIOCON",
    "SHANTIGEAR",
    "PNBHOUSING",
    "GLAXO",
    "INOXWIND",
    "MPHASIS",
    "AWL",
    "UJJIVANSFB",
    "HAVELLS",
    "JSFB",
    "SHANTIGOLD",
    "MANKIND",
    "TATATECH",
    "ACUTAAS",
    "MARICO",
    "AEROFLEX"
]

def fetch_nse_universe():
    df, src = None, None
    sess = requests.Session()
    sess.headers.update(HEADERS)
    try:            # warm up cookies — NSE's WAF rejects cookie-less requests from many IPs
        sess.get("https://www.nseindia.com/", timeout=15)
    except Exception:
        pass
    for url in NSE_LIST_URLS:
        try:
            r = sess.get(url, timeout=30)
            r.raise_for_status()
            if "<html" in r.text[:200].lower():          # WAF "Access Denied" page with HTTP 200
                raise RuntimeError("blocked by NSE WAF (HTML instead of CSV)")
            df = pd.read_csv(io.StringIO(r.text))
            df.columns = [c.strip() for c in df.columns]
            df = df[df["SYMBOL"].notna() & (df["SYMBOL"].astype(str).str.len() > 0)]
            df = df[["SYMBOL", "NAME OF COMPANY"]].copy()
            df["SYMBOL"] = df["SYMBOL"].astype(str).str.upper().str.strip()
            df = df.drop_duplicates("SYMBOL").reset_index(drop=True)
            src = "NSE official archive"
            print(f"  ✓ fetched full NSE list ({len(df):,} symbols) from {url.split('/')[2]}")
            break
        except Exception as exc:
            print(f"  ⚠ NSE list fetch failed on {url.split('/')[2]} "
                  f"({exc.__class__.__name__}: {exc})")
    if df is None:
        print(f"  → falling back to embedded list of {len(FALLBACK_SYMBOLS)} liquid names")
        df = pd.DataFrame({"SYMBOL": FALLBACK_SYMBOLS, "NAME OF COMPANY": ""})
        src = "embedded fallback"
    df["Yahoo"] = df["SYMBOL"].str.upper() + ".NS"
    df["source"] = src
    return df

UNIVERSE_ALL = fetch_nse_universe()
UNIVERSE_ALL.head(8)

## 3. Rank the universe — NSE Top 1000 by average daily turnover
One month of data is pulled for **every** NSE stock in chunked batches (with retries + backoff so a
rate-limit 429 never kills the run), each symbol's average daily turnover (₹ Cr) over the last 10
sessions is computed, and the top `CFG.universe_size` become the screening universe.

## 4. Full history download + data-health report
~1 year of daily OHLCV for the universe, then a **latest-session backfill**, then a health gate:
- **Yahoo data-lag repair:** right around/after the close, Yahoo often publishes the newest NSE
  bar with `Close = NaN` (Open/High/Low present). Those bars are useless for a close-based
  screener. This cell detects them and **rebuilds the candle from 15-min intraday data**
  (close = final trade of the session) — no tickers are lost to the lag.
- tickers with < `min_bars` bars → dropped
- **tickers still without a candle for the market's latest session → dropped** (suspended or
  genuinely no data) — only latest-session candles are screened (set
  `CFG.require_latest_session = False` to relax)
- OHLC anomalies (bad ticks) → auto-repaired (High = max(O,C,H), Low = min(O,C,L))

In [ ]:
# ============================================================
# 3. TURNOVER RANKING — NSE top N liquid stocks
# ============================================================
def chunked_download(symbols, period, chunk=None, label="download"):
    """Chunked yfinance download with per-chunk retry + exponential backoff.

    Returns (dict {yahoo_symbol: OHLCV DataFrame}, list_of_failed_symbols).
    A single bad chunk never aborts the rest of the run.
    """
    chunk = chunk or CFG.download_chunk
    out, failed = {}, []
    n_chunks = (len(symbols) + chunk - 1) // chunk
    for ci in range(n_chunks):
        part = symbols[ci * chunk:(ci + 1) * chunk]
        res, ok = None, False
        for attempt in range(1, CFG.max_retries + 1):
            try:
                res = yf.download(part, period=period, interval="1d", group_by="ticker",
                                  auto_adjust=True, threads=True, progress=False)
                if res is None or len(res) == 0:
                    raise RuntimeError("empty response")
                ok = True
                break
            except Exception as exc:
                wait = CFG.backoff_secs * (2 ** (attempt - 1)) + random.uniform(0, 2)
                print(f"  ⚠ {label} chunk {ci+1}/{n_chunks} attempt {attempt} failed "
                      f"({exc.__class__.__name__}) — retry in {wait:.0f}s")
                time.sleep(wait)
        if not ok:
            failed.extend(part)
            print(f"  ✖ {label} chunk {ci+1}/{n_chunks}: all {CFG.max_retries} attempts failed "
                  f"({len(part)} symbols skipped)")
            continue
        for sym in part:
            try:
                df = res.xs(sym, axis=1, level="Ticker")
            except (KeyError, TypeError, ValueError):
                continue  # symbol missing from response (delisted/renamed/404)
            df = df[["Open", "High", "Low", "Close", "Volume"]].copy()
            df = df.dropna(subset=["Open", "High", "Low", "Close"])
            if len(df) and np.isfinite(df["Close"].to_numpy(dtype=float)).all():
                out[sym] = df
        if (ci + 1) % 5 == 0 or ci == n_chunks - 1:
            print(f"  {label}: {min((ci+1)*chunk, len(symbols))}/{len(symbols)} symbols "
                  f"({len(out):,} good so far)")
        time.sleep(0.6)
    return out, failed

print(f"Ranking pass: 1 month of data for all {len(UNIVERSE_ALL):,} NSE symbols …")
t0 = time.time()
RANK_DATA, rank_failed = chunked_download(UNIVERSE_ALL["Yahoo"].tolist(), period="1mo",
                                          chunk=120, label="ranking")
print(f"  fetched {len(RANK_DATA):,}/{len(UNIVERSE_ALL):,} in {time.time()-t0:.0f}s "
      f"({len(rank_failed)} unavailable on Yahoo)")

rows = []
for sym, df in RANK_DATA.items():
    if len(df) < CFG.rank_bars_needed:
        continue
    tail = df.tail(10)
    rows.append({
        "Yahoo": sym,
        "Avg_Turnover_Cr": float((tail["Close"] * tail["Volume"]).mean() / 1e7),
        "Last_Close": float(df["Close"].iloc[-1]),
    })
if not rows:
    raise SystemExit("✖ Turnover ranking produced no data — Yahoo may be rate-limiting this IP. "
                     "Wait a minute and re-run this cell.")
rank_df = (pd.DataFrame(rows)
            .sort_values("Avg_Turnover_Cr", ascending=False)
            .reset_index(drop=True))
rank_df = rank_df[rank_df["Avg_Turnover_Cr"] >= CFG.min_avg_turnover_cr]
UNIVERSE = rank_df.head(CFG.universe_size).copy()
UNIVERSE = UNIVERSE.merge(UNIVERSE_ALL[["Yahoo", "SYMBOL", "NAME OF COMPANY"]],
                          on="Yahoo", how="left")

print(f"\nUniverse locked: top {len(UNIVERSE):,} of {len(rank_df):,} liquid NSE stocks "
      f"(turnover >= ₹{CFG.min_avg_turnover_cr} Cr/day)")
UNIVERSE[["SYMBOL", "NAME OF COMPANY", "Last_Close", "Avg_Turnover_Cr"]].head(10)

In [ ]:
# ============================================================
# 4. FULL HISTORY + LATEST-SESSION BACKFILL + DATA HEALTH
# ============================================================
import datetime as _dt

def backfill_latest(symbols, target_date, chunk=40):
    """Rebuild missing latest-session daily candles from 15-min data (Close = final trade).

    Yahoo frequently leaves Close=NaN on the newest NSE daily bar; the 15-min feed has the
    full session. Daily O/H/L and the 15-min aggregates agree to tick precision.
    """
    fixed = {}
    if not symbols:
        return fixed
    end = str(target_date + _dt.timedelta(days=1))
    n_chunks = (len(symbols) + chunk - 1) // chunk
    for ci in range(n_chunks):
        part = symbols[ci * chunk:(ci + 1) * chunk]
        intr, ok = None, False
        for attempt in range(1, CFG.max_retries + 1):
            try:
                intr = yf.download(part, start=str(target_date), end=end, interval="15m",
                                   group_by="ticker", auto_adjust=False,
                                   threads=True, progress=False)
                if intr is None or len(intr) == 0:
                    raise RuntimeError("empty response")
                ok = True
                break
            except Exception as exc:
                wait = CFG.backoff_secs * (2 ** (attempt - 1)) + random.uniform(0, 2)
                print(f"  ⚠ backfill chunk {ci+1}/{n_chunks} attempt {attempt} failed "
                      f"({exc.__class__.__name__}) — retry in {wait:.0f}s")
                time.sleep(wait)
        if not ok:
            continue
        for sym in part:
            try:
                i = intr.xs(sym, axis=1, level="Ticker").dropna(subset=["Open", "High", "Low", "Close"])
            except (KeyError, TypeError, ValueError):
                continue
            i = i.sort_index()
            if not len(i):
                continue
            row = dict(Open=float(i["Open"].iloc[0]), High=float(i["High"].max()),
                       Low=float(i["Low"].min()), Close=float(i["Close"].iloc[-1]),
                       Volume=float(i["Volume"].sum()))
            fixed[sym] = pd.DataFrame([row], index=pd.DatetimeIndex([pd.Timestamp(target_date)]))
        if (ci + 1) % 4 == 0 or ci == n_chunks - 1:
            print(f"  backfill: {min((ci+1)*chunk, len(symbols))}/{len(symbols)} — "
                  f"{len(fixed)} candles rebuilt")
        time.sleep(0.5)
    return fixed

print(f"Downloading {CFG.history_period} daily OHLCV for {len(UNIVERSE):,} symbols …")
t0 = time.time()
HIST, hist_failed = chunked_download(UNIVERSE["Yahoo"].tolist(), period=CFG.history_period,
                                     chunk=CFG.download_chunk, label="history")
print(f"  fetched {len(HIST):,}/{len(UNIVERSE):,} in {time.time()-t0:.0f}s")
if hist_failed:
    print(f"  ⚠ {len(hist_failed)} symbols failed after {CFG.max_retries} retries: "
          f"{hist_failed[:15]}{' …' if len(hist_failed) > 15 else ''}")
if not HIST:
    raise SystemExit("✖ Full history download failed completely — check internet and re-run this cell.")

max_last = max(df.index[-1].date() for df in HIST.values())

if CFG.backfill_latest_close:
    missing = [s for s, df in HIST.items() if df.index[-1].date() < max_last]
    if missing:
        print(f"\n{len(missing)} symbols lack a complete candle for {max_last} "
              f"(Yahoo NaN-close lag) — rebuilding from 15-min data …")
        t0 = time.time()
        fixed = backfill_latest(missing, max_last)
        for sym, bar in fixed.items():
            HIST[sym] = pd.concat([HIST[sym], bar])
        print(f"  rebuilt {len(fixed)}/{len(missing)} latest-session candles in {time.time()-t0:.0f}s")
    else:
        print(f"\nAll {len(HIST):,} symbols have a complete candle for {max_last} — no backfill needed.")

# ---- data health ----
DATA, short_list, stale_list = {}, [], []
for sym, df in HIST.items():
    if len(df) < CFG.min_bars:
        short_list.append(sym); continue
    if CFG.require_latest_session:
        if df.index[-1].date() < max_last:
            stale_list.append(sym); continue   # missing latest session (Yahoo data lag)
    elif (max_last - df.index[-1].date()).days > CFG.max_staleness_days:
        stale_list.append(sym); continue
    DATA[sym] = df

repaired = 0
for sym, df in DATA.items():
    bad = ((df["High"] < df["Low"])
           | (df["High"] < df[["Open", "Close"]].max(axis=1))
           | (df["Low"] > df[["Open", "Close"]].min(axis=1)))
    if bad.any():
        df["High"] = df[["Open", "Close", "High"]].max(axis=1)
        df["Low"] = df[["Open", "Close", "Low"]].min(axis=1)
        repaired += 1

print("\n── DATA HEALTH REPORT ───────────────────────────────────")
print(f"  Market data as-of date   : {max_last}")
print(f"  Screenable tickers       : {len(DATA):,} / {len(UNIVERSE):,}")
print(f"  Dropped — short history  : {len(short_list)}")
print(f"  Dropped — no candle of latest session (Yahoo lag/suspended): {len(stale_list)}")
print(f"  OHLC anomalies repaired  : {repaired}")
print("──────────────────────────────────────────────────────────")

## 5. Sweep engine
- **Swing lows:** 5-bar fractal, tie-aware (`low[i]` strictly below the 2 bars left, `<=` the 2 bars right — so exact double bottoms keep their first touch as the level).
- **Liquidity pools:** fractal levels within 0.15% of each other are merged into one pool
  (same liquidity). Each pool keeps its *birth* bar and *last touch* bar.
- A pool **sweeps** on the latest candle T when: low pierces below, close reclaims above (with
  buffers), depth ≤ `max_sweep_depth`, wick ratios pass, close is bullish and the prior close
  was above the level. If several pools match, the shallowest / newest is reported.
- **Score (0–100):** wick quality 25 · close position 15 · sweep depth 15 · volume 15 · RSI 15 · EMA trend 15.

In [ ]:
# ============================================================
# 5. SWEEP ENGINE
# ============================================================
def find_swing_lows(low, k):
    """Indices i where low[i] is a fractal swing low of low[i-k .. i+k].

    Tie-aware: strictly lower than the k bars on the LEFT, lower-or-EQUAL to the k bars
    on the RIGHT. An exact double/triple bottom (equal lows — common on NSE's 0.05 tick)
    keeps its FIRST touch as the level instead of being discarded entirely.
    Windows containing non-finite lows are skipped (bad data can't mint levels).
    """
    n = len(low)
    out = []
    for i in range(k, n - k):
        w = low[i - k:i + k + 1]
        if not np.isfinite(w).all():
            continue
        left, right = low[i - k:i], low[i + 1:i + k + 1]
        if low[i] < left.min() and low[i] <= right.min():
            out.append((i, float(low[i])))
    return out

def merge_pools(swings, dedup_pct):
    """Merge nearly-equal swing levels into liquidity pools: [level, first_bar, last_bar]."""
    pools = []
    for i, L in swings:
        for p in pools:
            if abs(L - p[0]) / p[0] <= dedup_pct:
                p[0] = min(p[0], L)
                p[2] = max(p[2], i)
                break
        else:
            pools.append([L, i, i])
    return pools

def rsi_wilder(close, period=14):
    """Wilder RSI series (aligned to close[1:])."""
    delta = np.diff(close)
    gain = pd.Series(np.clip(delta, 0, None)).ewm(alpha=1 / period, adjust=False).mean()
    loss = pd.Series(np.clip(-delta, 0, None)).ewm(alpha=1 / period, adjust=False).mean()
    out = pd.Series(np.where(loss > 0, 100 - 100 / (1 + gain / loss.where(loss > 0)),
                             np.where(gain > 0, 100.0, 50.0)), index=gain.index)
    return out   # zero-loss+zero-gain (flat tape) is neutral 50, not a fake 100

def analyze_symbol(df, cfg):
    """Return a setup record if the latest candle of df completed a valid swing-low sweep, else None."""
    n = len(df)
    if n < max(cfg.min_bars, 2 * cfg.fractal_k + 4):
        return None
    o = df["Open"].to_numpy(dtype=float)
    h = df["High"].to_numpy(dtype=float)
    l = df["Low"].to_numpy(dtype=float)
    c = df["Close"].to_numpy(dtype=float)
    v = np.nan_to_num(df["Volume"].to_numpy(dtype=float))   # Yahoo lags Volume on fresh bars;
                                                             # NaN here poisons Vol_x & turnover
    T = n - 1
    if not (np.isfinite(o[T]) and np.isfinite(h[T]) and np.isfinite(l[T])
            and np.isfinite(c[T]) and h[T] > 0):
        return None      # any non-finite OHLC on the signal bar -> NaN passes every
                         # comparison filter silently and mints a NaN-stop "setup"
    if c[T] < cfg.min_price:
        return None
    rng = h[T] - l[T]
    if rng <= 0:
        return None
    lw = min(o[T], c[T]) - l[T]

    pools = merge_pools(find_swing_lows(l, cfg.fractal_k), cfg.level_dedup_pct)
    best = None
    for level, first_i, last_i in pools:
        age = T - first_i
        if age <= 0 or age > cfg.level_lookback:
            continue
        if l[T] >= level * (1 - cfg.min_pierce_pct):          # no genuine pierce
            continue
        if c[T] <= level * (1 + cfg.close_above_buffer):       # no reclaim above
            continue
        depth = (level - l[T]) / level
        if depth > cfg.max_sweep_depth:                        # too deep = breakdown
            continue
        if lw / rng < cfg.min_wick_ratio:                      # no proper wick
            continue
        if lw < cfg.min_wick_pct_price * level:
            continue
        if cfg.require_bullish_close and c[T] <= o[T]:
            continue
        if cfg.require_prior_above and c[T - 1] <= level:      # was already below → not a sweep
            continue
        key = (depth, -age)
        if best is None or key < (best["depth"], -best["age"]):
            best = dict(level=level, age=age, touch_ago=T - last_i, depth=depth)
    if best is None:
        return None

    vol_base = float(v[T - cfg.volume_lookback:T].mean())
    vol_x = float(v[T]) / vol_base if vol_base > 0 else 1.0
    rsi = float(rsi_wilder(c, cfg.rsi_period).iloc[-1])
    e20 = float(pd.Series(c).ewm(span=20, adjust=False).mean().iloc[-1])
    e50 = float(pd.Series(c).ewm(span=50, adjust=False).mean().iloc[-1])
    close_pos = (c[T] - l[T]) / rng
    wick_ratio = lw / rng
    turnover_cr = float((c * v)[T - cfg.volume_lookback:T].mean() / 1e7)
    stop = float(l[T])
    target_ref = max(float(np.max(h[T - 20:T])), float(c[T]) * 1.01)
    rr = max(0.0, (target_ref - c[T]) / max(c[T] - stop, 1e-9))

    # ---- score components (each 0..1) ----
    s_wick = min(wick_ratio / 0.75, 1.0)
    s_close = max(0.0, min(1.0, (close_pos - 0.5) / 0.45))
    s_depth = max(0.0, 1.0 - best["depth"] / cfg.max_sweep_depth)
    if vol_x <= 1.0:
        s_vol = 0.5 * vol_x
    elif vol_x <= 3.0:
        s_vol = 0.5 + 0.5 * (vol_x - 1.0) / 2.0
    else:
        s_vol = max(0.0, 1.0 - (vol_x - 3.0) / 5.0)
    if 25.0 <= rsi <= 65.0:
        s_rsi = 1.0
    elif rsi < 25.0:
        s_rsi = max(0.0, 0.6 * rsi / 25.0)
    else:
        s_rsi = max(0.0, 1.0 - (rsi - 65.0) / 30.0)
    s_ema = {2: 1.0, 1: 0.55, 0: 0.2}[int(c[T] > e50) + int(c[T] > e20)]  # int() casts: numpy-bool "+" is logical OR, never 2

    score = 25 * s_wick + 15 * s_close + 15 * s_depth + 15 * s_vol + 15 * s_rsi + 15 * s_ema

    return dict(
        Yahoo=df.attrs.get("yahoo", ""),
        Date=str(df.index[-1].date()),
        Open=float(o[T]), Close=float(c[T]), High=float(h[T]), Low=float(stop),
        Swept_Level=float(best["level"]),
        Level_Age_Bars=int(best["age"]),
        Touch_Ago_Bars=int(best["touch_ago"]),
        Sweep_Depth_=float(best["depth"]) * 100.0,
        Above_Level_=(c[T] / best["level"] - 1.0) * 100.0,
        LowerWick_=wick_ratio * 100.0,
        ClosePos_=close_pos * 100.0,
        Vol_x=vol_x, RSI14=rsi,
        E20=bool(c[T] > e20), E50=bool(c[T] > e50),
        Avg_Turn_Cr=turnover_cr,
        Stop_Loss=stop, Target_Ref=target_ref, RR=rr,
        Score=float(score),
    )

def screen_all(data, cfg):
    """Run analyze_symbol over every ticker; return a DataFrame sorted by Score desc."""
    rows, errors = [], 0
    for sym, df in data.items():
        try:
            rec = analyze_symbol(df, cfg)
        except Exception:
            errors += 1
            continue
        if rec is not None:
            rec["Yahoo"] = sym
            rows.append(rec)
    if errors:
        print(f"  (engine skipped {errors} tickers on internal errors)")
    if not rows:
        return pd.DataFrame()
    res = pd.DataFrame(rows).sort_values("Score", ascending=False).reset_index(drop=True)
    res.insert(0, "#", range(1, len(res) + 1))
    return res

print("Engine loaded — find_swing_lows / merge_pools / analyze_symbol / screen_all")

In [ ]:
# ============================================================
# 5b. ENGINE SELF-TEST — synthetic candles (proves the logic)
# ============================================================
def _synthetic_df(rows):
    idx = pd.bdate_range("2026-01-05", periods=len(rows))
    return pd.DataFrame(rows, index=idx, columns=["Open", "High", "Low", "Close", "Volume"])

rows = []
for i in range(60):                                   # calm drift around 105
    o = 105.0 + (i % 5) * 0.2
    rows.append((o, o + 0.8, o - 0.8, o + 0.3, 1_000_000))
seq = [(104.2, 104.6, 103.4, 103.6, 1_000_000),       # decline into the swing low
       (103.6, 103.9, 102.1, 102.4, 1_000_000),
       (102.4, 102.6, 100.9, 101.2, 1_000_000),
       (101.2, 101.5, 100.0, 100.6, 1_200_000),       # <-- swing low 100.00 (fractal)
       (100.6, 101.4, 100.5, 101.1, 1_100_000),
       (101.1, 101.8, 100.9, 101.6, 1_000_000),
       (101.6, 102.4, 101.4, 102.1, 1_000_000),
       (102.1, 102.9, 101.9, 102.6, 1_100_000),
       (102.6, 103.1, 102.2, 102.8, 1_000_000)]       # bars 60..68
rows += seq
# bar 69: THE SWEEP — wick to 99.35 below level 100, close 102.6 back above
rows.append((101.8, 103.0, 99.35, 102.6, 1_800_000))

rec = analyze_symbol(_synthetic_df(rows), SweepConfig())
assert rec is not None, "positive test FAILED — sweep not detected"
assert abs(rec["Swept_Level"] - 100.0) < 1e-9, "wrong level picked"

rows_bad = rows[:-1] + [(101.8, 103.0, 99.35, 99.8, 1_800_000)]   # close BELOW level
assert analyze_symbol(_synthetic_df(rows_bad), SweepConfig()) is None, \
    "negative test FAILED — close below level must not count"

rows_flat = rows[:-1] + [(101.8, 103.0, 100.3, 102.6, 1_800_000)] # no pierce at all
assert analyze_symbol(_synthetic_df(rows_flat), SweepConfig()) is None, \
    "negative test 2 FAILED — no pierce must not count"

# regression: EQUAL-LOW double bottom (NSE tick size makes exact ties common) must be detected
rows_dbl = rows[:60] + [(104.0, 104.3, 102.5, 102.8, 1e6),
                        (102.8, 103.0, 100.0, 100.8, 1e6),    # low #1 = 100.00
                        (100.8, 101.9, 100.6, 101.6, 1e6),
                        (101.6, 101.9, 100.0, 100.9, 1e6),    # low #2 = 100.00 (exact tie)
                        (100.9, 102.2, 100.7, 102.0, 1e6),
                        (102.0, 102.8, 101.8, 102.5, 1e6),
                        (102.5, 103.0, 102.1, 102.7, 1e6),
                        (102.0, 103.2, 99.60, 102.4, 1.6e6)]  # sweep of the double bottom
rec_dbl = analyze_symbol(_synthetic_df(rows_dbl), SweepConfig())
assert rec_dbl is not None and abs(rec_dbl["Swept_Level"] - 100.0) < 1e-9, \
    "regression FAILED — equal-low double bottom must be sweepable"

# regression: NaN Low on the signal bar must NEVER produce a setup
rows_nan = rows[:-1] + [(101.8, 103.0, float("nan"), 102.6, 1.8e6)]
assert analyze_symbol(_synthetic_df(rows_nan), SweepConfig()) is None, \
    "regression FAILED — NaN low must be rejected, not scored"

# regression: flat tape RSI is neutral, not 100
assert abs(float(rsi_wilder(np.array([100.0] * 30)).iloc[-1]) - 50.0) < 1e-9, \
    "regression FAILED — flat-tape RSI must be 50"

print("✅ ENGINE SELF-TEST PASSED")
print(f"   positive  : level={rec['Swept_Level']:.2f} age={rec['Level_Age_Bars']}bars "
      f"depth={rec['Sweep_Depth_']:.2f}% wick={rec['LowerWick_']:.0f}% score={rec['Score']:.1f}")
print("   negative  : close-below-level rejected ✓ | no-pierce rejected ✓")
print("   regression: equal-low double bottom ✓ | NaN-low bar ✓ | flat-tape RSI ✓")

## 6. Run the screener
Screens every healthy ticker; a **setup exists only if the latest daily candle completed a sweep**.
Table sorted by Score. `E50` = close above EMA50 (trend context).

In [ ]:
# ============================================================
# 6. RUN
# ============================================================
t0 = time.time()
RESULTS = screen_all(DATA, CFG)
print(f"Screened {len(DATA):,} symbols in {time.time()-t0:.1f}s")

if RESULTS.empty:
    print("\n⚠ No liquidity-sweep setups on the latest daily candle.")
    print("\n  Try:  (a) run after today's 15:35 IST close (confirmed candle)")
    print("         (b) loosen rules, e.g.")
    print("              CFG.min_wick_ratio   = 0.25")
    print("              CFG.max_sweep_depth  = 0.03")
    print("              CFG.require_prior_above = False")
    print("              CFG.level_lookback   = 150")
else:
    res = RESULTS.copy()
    res["Name"] = res["Yahoo"].map(UNIVERSE.set_index("Yahoo")["NAME OF COMPANY"].to_dict()).fillna("")
    res["Symbol"] = res["Yahoo"].str.replace(".NS", "", regex=False)
    show = res.copy()
    fmt = {"Close": "{:.2f}", "Swept_Level": "{:.2f}", "Sweep_Depth_": "{:.2f}",
           "Above_Level_": "{:.2f}", "LowerWick_": "{:.1f}", "ClosePos_": "{:.1f}",
           "Vol_x": "{:.1f}", "RSI14": "{:.0f}", "Avg_Turn_Cr": "{:.1f}",
           "Stop_Loss": "{:.2f}", "Target_Ref": "{:.2f}", "RR": "{:.2f}", "Score": "{:.1f}"}
    for col, f in fmt.items():
        show[col] = show[col].map(f.format)
    cols = ["#", "Symbol", "Name", "Date", "Close", "Swept_Level", "Level_Age_Bars",
            "Sweep_Depth_", "Above_Level_", "LowerWick_", "ClosePos_", "Vol_x",
            "RSI14", "E50", "Avg_Turn_Cr", "Stop_Loss", "Target_Ref", "RR", "Score"]
    show = show[cols]
    show.columns = ["#", "Symbol", "Name", "Date", "Close", "Swept Low", "Age(bars)",
                    "Depth %", "Above %", "Wick %", "ClosePos %", "Vol x",
                    "RSI", ">EMA50", "Turn ₹Cr", "Stop", "Target", "R:R", "Score"]
    print(f"\n✅  {len(res)} SWING-BUY SWEEP SETUPS  —  candle of {res['Date'].iloc[0]}  (by score)\n")
    display(show)

In [ ]:
# ============================================================
# 7. TOP PICKS — detailed report + chart
# ============================================================
def setup_report(r):
    print("=" * 74)
    print(f"  {r.Symbol}  ({r.Name})   ·   close {r.Close:.2f} on {r.Date}")
    print("=" * 74)
    print(f"  Swept swing low : {r.Swept_Level:.2f}   (born {r.Level_Age_Bars} bars ago, "
          f"last touched {r.Touch_Ago_Bars} bars ago)")
    print(f"  Wick pierce     : low {r.Sweep_Depth_:.2f}% below level  →  close {r.Above_Level_:.2f}% ABOVE level")
    print(f"  Wick quality    : lower wick {r.LowerWick_:.0f}% of range · close at {r.ClosePos_:.0f}% of range")
    print(f"  Context         : volume {r.Vol_x:.1f}x 20d avg · RSI(14) {r.RSI14:.0f} · "
          f"above EMA20: {r.E20} · above EMA50: {r.E50}")
    print(f"  Trade plan      : entry ≤ {r.Close:.2f} (or next open) | SL below {r.Stop_Loss:.2f} | "
          f"ref target {r.Target_Ref:.2f} (R:R {r.RR:.1f})")

def plot_setup(sym, level, last_bars=45):
    df = DATA[sym].tail(last_bars)
    x = np.arange(len(df))
    fig, ax = plt.subplots(figsize=(11, 5.8))
    for i, (oo, hh, ll, cc) in enumerate(zip(df["Open"], df["High"], df["Low"], df["Close"])):
        col = "#1a7f37" if cc >= oo else "#c62828"
        ax.vlines(i, ll, hh, color=col, lw=0.9)
        ax.bar(i, max(abs(cc - oo), 0.01), bottom=min(oo, cc), width=0.62,
               color=col, edgecolor=col, lw=0.5)
    ax.axhline(level, color="#e65100", ls="--", lw=1.4, label=f"swept swing low = {level:.2f}")
    ax.axhline(df["Low"].iloc[-1], color="#6a1b9a", ls=":", lw=1.1,
               label=f"sweep low / SL zone = {df['Low'].iloc[-1]:.2f}")
    ax.annotate("SWEEP ✓", xy=(len(df) - 1, df["High"].iloc[-1]),
                xytext=(len(df) - 7, df["High"].iloc[-1]),
                fontsize=10, weight="bold", color="#e65100")
    step = max(len(df) // 8, 1)
    ax.set_xticks(x[::step])
    ax.set_xticklabels([d.strftime("%d %b") for d in df.index[::step]], rotation=25, fontsize=8)
    ax.set_title(f"{sym}  ·  daily liquidity sweep of swing low  ·  {df.index[-1].date()}",
                 fontsize=12, weight="bold")
    ax.grid(alpha=0.25)
    ax.legend(loc="upper left", fontsize=9)
    plt.tight_layout()
    plt.show()

if not RESULTS.empty:
    top = RESULTS.head(3).copy()
    name_map = UNIVERSE.set_index("Yahoo")["NAME OF COMPANY"].to_dict()
    top["Name"] = top["Yahoo"].map(name_map).fillna("")
    top["Symbol"] = top["Yahoo"].str.replace(".NS", "", regex=False)
    for _, r in top.iterrows():
        setup_report(r)
        plot_setup(r.Yahoo, r.Swept_Level)
        print()
else:
    print("No setups to chart today.")

## 8. Reading the results — swing-trading playbook
| Column | How to use it |
|--------|---------------|
| **Swept Low** | The liquidity level that got taken out. This is your *narrative*: stops below it were stopped, and the market closed back above. |
| **Age (bars)** | How old the level is. New (≤10) = recent structure; old (>40) = institutional shelf. Both valid. |
| **Depth %** | How far the wick ran below the level. Shallow (0.1–1%) = clean tap; near 2.5% = aggressive hunt — needs the other columns to be strong. |
| **Above %** | Margin of the close above the level. Bigger = stronger rejection. |
| **Wick %** | Lower wick as % of range. ≥50% is a textbook sweep candle. |
| **ClosePos %** | Where the close sits in the range. Near 100 = closed at the high (strong). |
| **Vol x** | Volume vs 20-day average. 1.2–3× confirms participation; >5× can be distribution — be selective. |
| **RSI / >EMA50** | Context. Sweeps *with* the trend (close > EMA50) perform best; sweeps in a downtrend are counter-trend reversals — smaller size, tighter stop. |
| **Stop / Target / R:R** | Stop = sweep low (wick low) — if it's violated, the sweep failed. Target = recent swing-high reference. Take partials at 1R, trail the rest. |

**Execution:** buy at the close of the sweep candle (aggressive) or next session (conservative).
**Invalidation:** next-day close below the swept level → exit, no questions.
**Best quality bar:** Score ≥ 70 + Vol x ≥ 1.2 + >EMA50 = True.

## 9. Troubleshooting & data notes
- **Yahoo NaN-close lag on the newest bar:** repaired automatically by the 15-min backfill
  (see the "rebuilt … latest-session candles" line). If some tickers still end up in
  "no candle of latest session", they either had no trades or the 15-min feed was also missing
  them — re-running the download cell a little later usually fixes it.
- **HTTP 429 (rate limit):** built-in retry with exponential backoff + jitter. If an entire chunk
  still fails, wait ~60 s and re-run just that cell — earlier cells' variables are retained.
- **NSE list blocked:** falls back to the embedded top-300 list automatically (check the printed source line).
- **Symbols "unavailable on Yahoo":** renamed/delisted tickers (e.g. corporate renames). They are
  skipped; the top-1000 ranking compensates by pulling in the next liquid name.
- **Intraday runs:** before 15:30 IST the latest candle is live and incomplete — sweeps can "disappear".
  For final lists run after **15:35 IST**.
- **Stale tickers** (suspension, new listings) are dropped in the data-health gate, not screened.
- **Data source:** Yahoo Finance daily bars (adjusted). Occasional bad ticks are auto-repaired
  (see the health report). If a specific stock looks off on NSE, check the chart cell.
- **Re-running:** re-run all cells to refresh. Total runtime is typically 5–10 minutes.